In [1]:
import os
import re
import json
import numpy as np
import pandas as pd
import jieba
from pathlib import Path
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from umap import UMAP
import hdbscan
from tqdm.auto import tqdm

# Stopword and tokenizer configurations
CHINESE_WORD = re.compile(r"^[\u4e00-\u9fff]{2,}$")

def normalize_llm_sentiment(value) -> str:
    label = str(value or "").strip().lower()
    if label in {"负向", "负面", "negative", "neg"} or "负" in label:
        return "negative"
    if label in {"正向", "正面", "positive", "pos"} or "正" in label:
        return "positive"
    if label in {"中立", "中性", "neutral", "neu"} or "中" in label:
        return "neutral"
    return "unknown"

def parse_count(value) -> float:
    if value is None or pd.isna(value):
        return 0.0
    text = str(value).replace(",", "").strip()
    if not text:
        return 0.0
    try:
        if text.endswith("万"):
            return float(text[:-1]) * 10000
        if text.endswith("亿"):
            return float(text[:-1]) * 100000000
        return float(text)
    except ValueError:
        return 0.0

def percentile_score(series: pd.Series) -> pd.Series:
    series = pd.to_numeric(series, errors="coerce").fillna(0)
    if series.nunique(dropna=False) <= 1:
        return pd.Series(50.0, index=series.index)
    return series.rank(method="average", pct=True) * 100

def _safe_cluster_metric(metric_func, embeddings_array, labels_array, **kwargs) -> float:
    labels_array = np.asarray(labels_array)
    valid_mask = labels_array != -1
    x = np.asarray(embeddings_array)[valid_mask]
    y = labels_array[valid_mask]
    unique_labels = np.unique(y)
    if len(unique_labels) < 2 or len(y) <= len(unique_labels):
        return np.nan
    try:
        return float(metric_func(x, y, **kwargs))
    except ValueError:
        return np.nan

def topic_diversity_score(topic_word_lists: list[list[str]]) -> float:
    all_words = [word for words in topic_word_lists for word in words]
    if not all_words:
        return np.nan
    return len(set(all_words)) / len(all_words)

def _default_tokenizer(text: str) -> list[str]:
    return [token for token in re.findall(r"\w+", str(text).lower()) if len(token) > 1]

In [1]:
MODEL_PATH = Path("embModel/Qwen3-Embedding-0.6B")
KEYWORD_FILTER_CONFIG_PATH = Path("config/station_keyword_filter.json")
STOPWORD_PATHS = [Path("stopword.txt"), Path("stopinthis.txt")]
RESULT_PATH = Path("output/0623/bertopic_kmeans_results.csv")
BATCH_SIZE = 16
MIN_TEXT_LENGTH = 4

In [1]:
# Load raw records
LLM_OUTPUT_CANDIDATES = [Path("llmOutput"), Path("llmPart/outputs"), Path("analysis_json")]
analysis_paths = []
LLM_OUTPUT_DIR = None

for candidate in LLM_OUTPUT_CANDIDATES:
    candidate_paths = sorted(candidate.glob("*/*_analysis.json"))
    if candidate_paths:
        LLM_OUTPUT_DIR = candidate
        analysis_paths = candidate_paths
        break

if not analysis_paths:
    searched = "、".join(str(path.resolve()) for path in LLM_OUTPUT_CANDIDATES)
    raise FileNotFoundError(f"没有找到 *_analysis.json 文件，已搜索：{searched}")

records = []
for analysis_path in analysis_paths:
    with analysis_path.open("r", encoding="utf-8") as f:
        station_records = json.load(f)
    station = analysis_path.parent.name
    for record in station_records:
        if isinstance(record, dict):
            records.append({**record, "station": station})

raw = pd.DataFrame(records)
print(f"读取分析目录：{LLM_OUTPUT_DIR.resolve()}；分析文件：{len(analysis_paths):,} 个；原始记录：{len(raw):,} 条")

读取分析目录：/Users/clsljt/Documents/hhgh/station/llmOutput；分析文件：10 个；原始记录：1,208 条


In [1]:
# 1. Load Stopwords and Keywords Filter Configuration
stop_words = {"无标题"}
for path in STOPWORD_PATHS:
    if path.is_file():
        stop_words.update(word.strip() for word in path.read_text(encoding="utf-8").splitlines() if word.strip())

with KEYWORD_FILTER_CONFIG_PATH.open("r", encoding="utf-8") as f:
    keyword_filter_config = json.load(f)

keyword_exclude_words = {word.strip() for word in keyword_filter_config.get("exclude_words", []) if str(word).strip()}
keyword_prefer_words = {word.strip() for word in keyword_filter_config.get("prefer_words", []) if str(word).strip()}
stop_words.update(keyword_exclude_words)

def tokenize_zh(text: str) -> list[str]:
    return [
        word.strip()
        for word in jieba.lcut(text)
        if len(word.strip()) > 1
        and not word.strip().isdigit()
        and word.strip() not in stop_words
    ]

def tokenize_chinese_only(text: str) -> list[str]:
    return [
        word.strip()
        for word in jieba.lcut(text)
        if CHINESE_WORD.fullmatch(word.strip()) and word.strip() not in stop_words
    ]

def topic_coherence_npmi(
    docs: list[str],
    topic_word_lists: list[list[str]],
    tokenizer=None,
    eps: float = 1e-12,
) -> float:
    tokenizer = tokenizer or tokenize_zh
    tokenized_docs = [set(tokenizer(doc)) for doc in docs]
    n_docs = len(tokenized_docs)
    if n_docs == 0:
        return np.nan

    doc_freq = {}
    for tokens in tokenized_docs:
        for token in tokens:
            doc_freq[token] = doc_freq.get(token, 0) + 1

    pair_scores = []
    for words in topic_word_lists:
        words = [word for word in dict.fromkeys(words) if doc_freq.get(word, 0) > 0]
        if len(words) < 2:
            continue
        for i, word_i in enumerate(words[:-1]):
            docs_i = doc_freq[word_i]
            for word_j in words[i + 1:]:
                docs_j = doc_freq[word_j]
                co_docs = sum(1 for tokens in tokenized_docs if word_i in tokens and word_j in tokens)
                p_i = docs_i / n_docs
                p_j = docs_j / n_docs
                p_ij = (co_docs + eps) / n_docs
                npmi = np.log(p_ij / (p_i * p_j)) / (-np.log(p_ij))
                pair_scores.append(npmi)

    return float(np.mean(pair_scores)) if pair_scores else np.nan

# 2. Filter raw negative data (among all 1208 records)
raw["llm_sentiment_group"] = raw["llm_sentiment"].map(normalize_llm_sentiment)
kmeans_keep = raw["llm_sentiment_group"].eq("negative")

neg_df = raw.loc[kmeans_keep].copy()
if "content" in neg_df.columns:
    neg_df = neg_df.drop(columns=["content"])

description_text = neg_df["description"].fillna("").astype(str).str.strip()
post_text = raw.loc[kmeans_keep, "content"].fillna("").astype(str).str.strip()
neg_df["source_content"] = post_text
neg_df["analysis_text"] = description_text.where(description_text.ne(""), post_text)

clean_content = neg_df["analysis_text"].fillna("").astype(str).str.strip()
keep = clean_content.ne("") & clean_content.ne("无标题") & clean_content.str.len().ge(MIN_TEXT_LENGTH)

kmeans_df = neg_df.loc[keep].rename(columns={"note_id": "id", "analysis_text": "content"}).reset_index(drop=True)
kmeans_documents = clean_content.loc[keep].tolist()

# 3. Calculate Qwen and Multilingual Embeddings
print(f"Loading Qwen embedding model from {MODEL_PATH}...")
try:
    import torch
    EMBEDDING_DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
except Exception:
    EMBEDDING_DEVICE = "cpu"

qwen_model = SentenceTransformer(str(MODEL_PATH), device=EMBEDDING_DEVICE, trust_remote_code=True, local_files_only=True)
qwen_embeddings = qwen_model.encode(
    kmeans_documents,
    batch_size=BATCH_SIZE,
    show_progress_bar=False,
    normalize_embeddings=True,
    convert_to_numpy=True,
)

print("Loading paraphrase-multilingual embedding model...")
try:
    default_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2", device=EMBEDDING_DEVICE)
except Exception:
    default_model = SentenceTransformer("embModel/BAAI/bge-small-zh-v1.5", device=EMBEDDING_DEVICE)

default_embeddings = default_model.encode(
    kmeans_documents,
    batch_size=BATCH_SIZE,
    show_progress_bar=False,
    normalize_embeddings=True,
    convert_to_numpy=True,
)

# Prep exp3 combined text
raw_neg = raw.loc[kmeans_keep]
title_text = raw_neg["title"].fillna("").astype(str).str.strip()
content_text = raw_neg["content"].fillna("").astype(str).str.strip()
combined_text = (title_text + " " + content_text).str.strip()
keep_comb = combined_text.ne("") & combined_text.ne("无标题") & combined_text.str.len().ge(MIN_TEXT_LENGTH)
combined_documents = combined_text.loc[keep_comb].tolist()

combined_embeddings = default_model.encode(
    combined_documents,
    batch_size=BATCH_SIZE,
    show_progress_bar=False,
    normalize_embeddings=True,
    convert_to_numpy=True,
)

# Common evaluator function (evaluates on all documents including noise -1)
def calculate_metrics(experiment_name, docs, embeddings, topics):
    labels_array = np.asarray(topics)
    topic_ids = np.unique(labels_array)
    
    topic_documents = (
        pd.DataFrame({"topic": topics, "content": docs})
        .groupby("topic", sort=True)["content"]
        .agg(" ".join)
    )

    label_vectorizer = TfidfVectorizer(tokenizer=tokenize_chinese_only, token_pattern=None, lowercase=False, min_df=1)
    label_matrix = label_vectorizer.fit_transform(topic_documents)
    label_terms = label_vectorizer.get_feature_names_out()

    topic_word_map = {}
    for topic_id, row in zip(topic_documents.index, label_matrix.toarray()):
        weighted_row = row.copy()
        for index, term in enumerate(label_terms):
            if term in keyword_prefer_words:
                weighted_row[index] *= 1.35

        top_indices = [index for index in weighted_row.argsort()[::-1] if row[index] > 0][:10]
        words = [label_terms[index] for index in top_indices]
        topic_word_map[topic_id] = words
        
    word_lists = []
    for topic_id in sorted(topic_ids):
        words = topic_word_map.get(topic_id, [])
        if words:
            word_lists.append(words)

    metrics = {
        "experiment": experiment_name,
        "n_documents": int(len(docs)),
        "n_topics": int(len(topic_ids)),
        "silhouette_score": float(silhouette_score(embeddings, labels_array, metric="cosine")),
        "davies_bouldin_index": float(davies_bouldin_score(embeddings, labels_array)),
        "calinski_harabasz_index": float(calinski_harabasz_score(embeddings, labels_array)),
        "topic_coherence_npmi": topic_coherence_npmi(docs, word_lists, tokenizer=tokenize_zh),
        "topic_diversity": topic_diversity_score(word_lists),
    }
    return metrics

Loading Qwen embedding model from embModel/Qwen3-Embedding-0.6B...
Loading paraphrase-multilingual embedding model...


## 0. 本文提出的核心方法（来自 v2.ipynb）：description列 + Qwen embedding + KMeans(K=10)

In [1]:
# Proposed Model: description + Qwen embedding + KMeans(K=10)
print("Running Proposed Model (v2)...")
cluster_model_proposed = KMeans(n_clusters=10, random_state=42, n_init=10)
umap_proposed = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine', random_state=42)
topic_model_proposed = BERTopic(umap_model=umap_proposed, hdbscan_model=cluster_model_proposed, vectorizer_model=CountVectorizer(tokenizer=tokenize_zh, token_pattern=None, min_df=1))
topics_proposed, _ = topic_model_proposed.fit_transform(kmeans_documents, qwen_embeddings)

metrics_proposed = calculate_metrics("kmeans_bertopic_k10 (v2 Proposed)", kmeans_documents, qwen_embeddings, topics_proposed)
pd.DataFrame([metrics_proposed])

Running Proposed Model (v2)...


## 1. 算法消融实验：使用现在的数据（describe列+Qwen embedding），进行HDBSCAN聚类

In [1]:
# Experiment 1: description + Qwen embedding + BERTopic(HDBSCAN)
print("Running Experiment 1...")
hdbscan_model_1 = hdbscan.HDBSCAN(min_cluster_size=5, min_samples=2, prediction_data=True)
umap_1 = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine', random_state=42)
topic_model_1 = BERTopic(umap_model=umap_1, hdbscan_model=hdbscan_model_1, vectorizer_model=CountVectorizer(tokenizer=tokenize_zh, token_pattern=None, min_df=1))
topics_1, _ = topic_model_1.fit_transform(kmeans_documents, qwen_embeddings)

metrics_1 = calculate_metrics("exp1_desc_qwen_hdbscan", kmeans_documents, qwen_embeddings, topics_1)
pd.DataFrame([metrics_1])

Running Experiment 1...


## 2. 向量模型消融实验：使用describe，进行HDBSCAN聚类（使用默认多语言Emb模型）

In [1]:
# Experiment 2: description + default emb model + BERTopic(HDBSCAN)
print("Running Experiment 2...")
hdbscan_model_2 = hdbscan.HDBSCAN(min_cluster_size=5, min_samples=2, prediction_data=True)
umap_2 = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine', random_state=42)
topic_model_2 = BERTopic(umap_model=umap_2, hdbscan_model=hdbscan_model_2, vectorizer_model=CountVectorizer(tokenizer=tokenize_zh, token_pattern=None, min_df=1))
topics_2, _ = topic_model_2.fit_transform(kmeans_documents, default_embeddings)

metrics_2 = calculate_metrics("exp2_desc_default_hdbscan", kmeans_documents, default_embeddings, topics_2)
pd.DataFrame([metrics_2])

Running Experiment 2...


## 3. 文本特征消融实验：使用title+content拼接，进行HDBSCAN聚类（使用默认多语言Emb模型）

In [1]:
# Experiment 3: title + content + default emb model + BERTopic(HDBSCAN)
print("Running Experiment 3...")
hdbscan_model_3 = hdbscan.HDBSCAN(min_cluster_size=5, min_samples=2, prediction_data=True)
# Focus UMAP locally by setting n_neighbors=5 to encourage cluster division for combined text
umap_3 = UMAP(n_neighbors=5, n_components=5, min_dist=0.0, metric='cosine', random_state=42)
topic_model_3 = BERTopic(umap_model=umap_3, hdbscan_model=hdbscan_model_3, vectorizer_model=CountVectorizer(tokenizer=tokenize_zh, token_pattern=None, min_df=1))
topics_3, _ = topic_model_3.fit_transform(combined_documents, combined_embeddings)

metrics_3 = calculate_metrics("exp3_combined_default_hdbscan", combined_documents, combined_embeddings, topics_3)
pd.DataFrame([metrics_3])

Running Experiment 3...


## 4. 对比实验结果汇总

In [1]:
# Summary
results = [
    metrics_proposed,
    metrics_1,
    metrics_2,
    metrics_3
]
df_res = pd.DataFrame(results)
# Save results
df_res.to_csv("output/0623/comp_experiment_results.csv", index=False)
df_res